# Check Unknown Cities in Missing Mappings

Cross-check if missing airport codes have 'Unknown' city assignments or actual city names.

In [1]:
import pandas as pd

# Load data
revenue_df = pd.read_parquet('gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet')
mapping_df = pd.read_excel('gs://agntworks-data-dev/wheelsup/raw/Mappings/ICAO to City Mapping.xlsx')

# Extract mapped codes
mapped_codes = set(mapping_df['displayCode'].dropna().unique())

# Get all revenue airports
from_counts = revenue_df['FromAirport'].value_counts()
to_counts = revenue_df['ToAirport'].value_counts()
all_revenue_airports = set(from_counts.index) | set(to_counts.index)

# Find missing airports
missing_airports = all_revenue_airports - mapped_codes

print(f"Total missing airports: {len(missing_airports)}")
print(f"Total revenue airports: {len(all_revenue_airports)}")

Total missing airports: 621
Total revenue airports: 2631


In [2]:
# Check FromAirport with missing codes
print("\n" + "="*70)
print("1. FROM AIRPORT ANALYSIS")
print("="*70)

from_missing_records = revenue_df[revenue_df['FromAirport'].isin(missing_airports)]
print(f"\nTotal records with missing FromAirport: {len(from_missing_records)}")
print(f"Unknown FromCity values: {(from_missing_records['FromCity'] == 'Unknown').sum()}")
print(f"Non-Unknown FromCity values: {(from_missing_records['FromCity'] != 'Unknown').sum()}")

print(f"\nBreakdown by FromCity:")
from_city_breakdown = from_missing_records['FromCity'].value_counts()
print(from_city_breakdown.head(20))


1. FROM AIRPORT ANALYSIS

Total records with missing FromAirport: 20514
Unknown FromCity values: 20514
Non-Unknown FromCity values: 0

Breakdown by FromCity:
FromCity
Unknown    20514
Name: count, dtype: int64


In [3]:
# Check ToAirport with missing codes
print("\n" + "="*70)
print("2. TO AIRPORT ANALYSIS")
print("="*70)

to_missing_records = revenue_df[revenue_df['ToAirport'].isin(missing_airports)]
print(f"\nTotal records with missing ToAirport: {len(to_missing_records)}")
print(f"Unknown ToCity values: {(to_missing_records['ToCity'] == 'Unknown').sum()}")
print(f"Non-Unknown ToCity values: {(to_missing_records['ToCity'] != 'Unknown').sum()}")

print(f"\nBreakdown by ToCity:")
to_city_breakdown = to_missing_records['ToCity'].value_counts()
print(to_city_breakdown.head(20))


2. TO AIRPORT ANALYSIS

Total records with missing ToAirport: 20622
Unknown ToCity values: 20622
Non-Unknown ToCity values: 0

Breakdown by ToCity:
ToCity
Unknown    20622
Name: count, dtype: int64


In [4]:
# Detailed view: missing airports and their city assignments
print("\n" + "="*70)
print("3. MISSING AIRPORTS WITH CITY ASSIGNMENTS")
print("="*70)

missing_airport_details = []
for airport in sorted(missing_airports):
    from_records = revenue_df[revenue_df['FromAirport'] == airport]
    to_records = revenue_df[revenue_df['ToAirport'] == airport]

    from_cities = from_records['FromCity'].unique()
    to_cities = to_records['ToCity'].unique()

    all_cities = set(from_cities) | set(to_cities)

    from_count = len(from_records)
    to_count = len(to_records)
    total = from_count + to_count

    has_unknown = 'Unknown' in all_cities

    missing_airport_details.append({
        'Airport': airport,
        'From': from_count,
        'To': to_count,
        'Total': total,
        'Cities': ', '.join(sorted(all_cities)),
        'Has Unknown': 'Yes' if has_unknown else 'No'
    })

missing_df = pd.DataFrame(missing_airport_details)
missing_df = missing_df.sort_values('Total', ascending=False)

print(f"\nTotal missing airports: {len(missing_df)}")
print(f"Missing airports with 'Unknown' city: {(missing_df['Has Unknown'] == 'Yes').sum()}")
print(f"Missing airports with actual city names: {(missing_df['Has Unknown'] == 'No').sum()}")

print(f"\n{missing_df.to_string(index=False)}")


3. MISSING AIRPORTS WITH CITY ASSIGNMENTS

Total missing airports: 621
Missing airports with 'Unknown' city: 621
Missing airports with actual city names: 0

Airport  From   To  Total  Cities Has Unknown
   KT82  1667 1727   3394 Unknown         Yes
   K5B2   746  767   1513 Unknown         Yes
   K27K   720  724   1444 Unknown         Yes
   K0A9   718  726   1444 Unknown         Yes
   KF45   611  594   1205 Unknown         Yes
   K29D   590  605   1195 Unknown         Yes
   K1R8   557  533   1090 Unknown         Yes
   K1A5   441  444    885 Unknown         Yes
   KF82   456  421    877 Unknown         Yes
   K5C1   421  427    848 Unknown         Yes
   KX60   379  452    831 Unknown         Yes
   K11R   387  397    784 Unknown         Yes
   K8A0   351  318    669 Unknown         Yes
   K46U   306  320    626 Unknown         Yes
   KM19   262  291    553 Unknown         Yes
   K4I3   264  272    536 Unknown         Yes
   KO69   247  270    517 Unknown         Yes
   K20V   253 

In [5]:
# Export to CSV
missing_df.to_csv('missing_airports_with_cities.csv', index=False)
print("✓ Detailed report exported to: missing_airports_with_cities.csv")

✓ Detailed report exported to: missing_airports_with_cities.csv


In [6]:
# Summary
print("\n" + "="*70)
print("4. SUMMARY")
print("="*70)

unknown_records = len(revenue_df[(revenue_df['FromCity'] == 'Unknown') | (revenue_df['ToCity'] == 'Unknown')])
total_records = len(revenue_df)
print(f"\nTotal 'Unknown' city records in revenue data: {unknown_records} ({100*unknown_records/total_records:.2f}%)")

missing_with_unknown = revenue_df[
    (revenue_df['FromAirport'].isin(missing_airports)) &
    (revenue_df['FromCity'] == 'Unknown')
]
print(f"Missing airport records marked as 'Unknown': {len(missing_with_unknown)}")

missing_with_cities = revenue_df[
    (revenue_df['FromAirport'].isin(missing_airports)) &
    (revenue_df['FromCity'] != 'Unknown')
]
print(f"Missing airport records with actual cities: {len(missing_with_cities)}")

print(f"\nRecords in missing airports overall: {len(from_missing_records) + len(to_missing_records)}")


4. SUMMARY

Total 'Unknown' city records in revenue data: 40744 (1.90%)
Missing airport records marked as 'Unknown': 20514
Missing airport records with actual cities: 0

Records in missing airports overall: 41136
